# Iris: Bayesian decision making (chuyển sang PyTorch)

Bản gốc **không dùng Keras**: ước lượng tham số bằng numpy và phân loại bằng `GaussianNB` của scikit-learn. Notebook này giữ sklearn làm mốc so sánh và thêm hai bản cài đặt bằng PyTorch:

1. Naive Bayes Gaussian (phương sai đường chéo), tương đương `GaussianNB`.
2. Bayes Gaussian đầy đủ ma trận hiệp phương sai (dùng tham số đã ước lượng ở bước 3 của bản gốc).

Quy tắc quyết định: chọn lớp c có `log p(x | c) + log P(c)` lớn nhất.

# (1) Thư viện

In [ ]:
import math, time
import numpy as np
import torch
from sklearn import datasets
from sklearn.naive_bayes import GaussianNB
from torch.distributions import MultivariateNormal

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
print(torch.__version__)

# (2) Chuẩn bị dữ liệu Iris

Tách 60% train (90 mẫu) và 40% test (60 mẫu) ngẫu nhiên, đặt seed để kết quả lặp lại được.

In [ ]:
iris = datasets.load_iris()
rng = np.random.RandomState(SEED)
train_indx = rng.choice(range(0, 150), int(iris.target.shape[0] / 100 * 60), replace=False)
test_indx = np.array([indx for indx in np.arange(0, 150, 1) if indx not in train_indx])

x_train, y_train = iris.data[train_indx, :], iris.target[train_indx]
x_test,  y_test  = iris.data[test_indx, :],  iris.target[test_indx]
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)   # (90,4) (90,) (60,4) (60,)

# sang tensor float64 để khớp độ chính xác của numpy/sklearn
Xtr, ytr = torch.tensor(x_train, dtype=torch.float64), torch.tensor(y_train)
Xte, yte = torch.tensor(x_test,  dtype=torch.float64), torch.tensor(y_test)

# (3) Ước lượng tham số (trung bình và hiệp phương sai từng lớp)

`np.mean/np.cov` → `tensor.mean` và `torch.cov` (cùng hiệu chỉnh n − 1).

In [ ]:
classes = torch.unique(ytr)
for c in classes:
    Xc = Xtr[ytr == c]
    print(f"--- Lớp {c.item()} ---")
    print("mu =", Xc.mean(0).numpy().round(4))
    print("sigma =\n", torch.cov(Xc.T).numpy().round(4))

# (4) Quyết định Bayes

## 4.1 Mốc so sánh: sklearn GaussianNB

In [ ]:
gnb = GaussianNB()
y_pred_sk = gnb.fit(x_train, y_train).predict(x_test)
print('Testing classification accuracy (sklearn): %.4f' % ((y_test == y_pred_sk).sum() / x_test.shape[0]))

## 4.2 Naive Bayes Gaussian bằng PyTorch

sklearn dùng phương sai MLE (chia n) cộng thêm `var_smoothing × max(phương sai từng đặc trưng)`; cài đặt dưới đây làm y hệt nên phải cho dự đoán trùng sklearn.

In [ ]:
class GaussianNBTorch:
    def __init__(self, var_smoothing=1e-9):
        self.vs = var_smoothing

    def fit(self, X, y):
        self.classes = torch.unique(y)
        self.mu = torch.stack([X[y == c].mean(0) for c in self.classes])                 # (C, D)
        var = torch.stack([X[y == c].var(0, unbiased=False) for c in self.classes])      # (C, D)
        self.var = var + self.vs * X.var(0, unbiased=False).max()
        self.log_prior = torch.log(torch.stack([(y == c).double().mean() for c in self.classes]))
        return self

    def joint_log_likelihood(self, X):
        diff2 = (X[:, None, :] - self.mu[None, :, :]) ** 2                               # (N, C, D)
        const = torch.log(2 * math.pi * self.var).sum(1)[None, :]                        # (1, C)
        ll = -0.5 * (const + (diff2 / self.var[None, :, :]).sum(2))                      # (N, C)
        return ll + self.log_prior[None, :]

    def predict(self, X):
        return self.classes[self.joint_log_likelihood(X).argmax(1)]

nb = GaussianNBTorch().fit(Xtr, ytr)
y_pred_nb = nb.predict(Xte)
print('Testing classification accuracy (PyTorch GaussianNB): %.4f' % (y_pred_nb == yte).double().mean().item())
print('Dự đoán trùng sklearn:', bool((y_pred_nb.numpy() == y_pred_sk).all()))
print('Trung bình trùng sklearn:', np.allclose(nb.mu.numpy(), gnb.theta_))
print('Phương sai trùng sklearn:', np.allclose(nb.var.numpy(), gnb.var_))

## 4.3 Bayes Gaussian đầy đủ ma trận hiệp phương sai (PyTorch)

Dùng đúng `mu` và `sigma` ở bước 3 (bản gốc có tính `sigma` nhưng `GaussianNB` không dùng đến). Cộng thêm 1e-9·I để tránh ma trận suy biến.

In [ ]:
class GaussianBayesFullCov:
    def fit(self, X, y):
        self.classes = torch.unique(y)
        self.dists = []
        for c in self.classes:
            Xc = X[y == c]
            cov = torch.cov(Xc.T) + 1e-9 * torch.eye(X.shape[1], dtype=X.dtype)
            self.dists.append(MultivariateNormal(Xc.mean(0), covariance_matrix=cov))
        self.log_prior = torch.log(torch.stack([(y == c).double().mean() for c in self.classes]))
        return self

    def predict(self, X):
        ll = torch.stack([d.log_prob(X) for d in self.dists], dim=1) + self.log_prior[None, :]
        return self.classes[ll.argmax(1)]

fc = GaussianBayesFullCov().fit(Xtr, ytr)
y_pred_fc = fc.predict(Xte)
print('Testing classification accuracy (PyTorch full covariance): %.4f' % (y_pred_fc == yte).double().mean().item())

# Nhận xét (điền sau khi có kết quả)

- Accuracy sklearn: …  | PyTorch GaussianNB: …  | PyTorch full covariance: …
- Bản PyTorch GaussianNB trùng sklearn (dự đoán, mu, phương sai): …
- Vì sao full covariance có thể khác: …

# Ghi chú

- Notebook gốc không dùng Keras; phần chuyển đổi là cài lại bộ phân loại Bayes bằng PyTorch và đối chiếu với sklearn.
- Với một lần tách ngẫu nhiên, accuracy phụ thuộc seed (bản gốc ghi 0.9667 ở một lần chạy không đặt seed).